# 02 — DCGAN Run 2: 280K Game Sprite Generation (75 Epochs Full Production)

**Project:** Prompt-to-Game-Asset Generator  
**Model:** Deep Convolutional GAN (DCGAN) — Run 2 Architecture  
**Target:** 75 Epochs (Full training run)  

### Key Improvements in Run 2:
1. **Anti-Checkerboard Upsampling**: Generator replaces transposed convolutions (`Conv2DTranspose`) with `UpSampling2D(size=(2,2), interpolation='nearest') + Conv2D(kernel=3)`. This eliminates stride-2 checkerboard artifacts and floating border noise.
2. **One-Sided Label Smoothing**: Real labels softened from 1.0 -> 0.9. Discriminator logits are prevented from exploding to +/- infinity.
3. **Configurable Instance Noise**: Soft Gaussian noise added to discriminator input tensors, smoothly annealing over early epochs to prevent discriminator over-saturation.
4. **Discriminator Monitoring**: Real prediction mean D(x) and fake prediction mean D(G(z)) are logged every epoch.
5. **Clean RGBA Transparency**: 64x64x4 RGBA output retained, respecting discrete sprite cutouts.
6. **Fresh Weights Guaranteed**: RUN_LABEL isolates checkpoints, configs, and output archives to avoid accidental restoration of Run 1 weights.


---
## Section 2: Environment and GPU Setup

In [ ]:
import os, sys, json, csv, time, random, hashlib, shutil, zipfile, platform, io
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
from pathlib import Path

print(f"Python      : {sys.version}")
print(f"TensorFlow  : {tf.__version__}")
print(f"Platform    : {platform.platform()}")
print(f"Timestamp   : {datetime.utcnow().isoformat()}Z")

# ── GPU detection ────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPUs found  : {len(gpus)}")
for g in gpus:
    print(f"  {g.name}")
    try:
        tf.config.experimental.set_memory_growth(g, True)
        print(f"    memory growth: enabled")
    except Exception as e:
        print(f"    memory growth: {e}")

# ── Distribution strategy ────────────────────────────────────────────────────
if len(gpus) >= 2:
    strategy = tf.distribute.MirroredStrategy()
    print(f"\nStrategy    : MirroredStrategy ({strategy.num_replicas_in_sync} replicas)")
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy('/GPU:0')
    print("\nStrategy    : OneDeviceStrategy (1 GPU)")
else:
    strategy = tf.distribute.OneDeviceStrategy('/CPU:0')
    print("\nWARNING: No GPU detected. CPU only — do NOT run full training on CPU.")
    print("Switch Kaggle accelerator to GPU T4 x2 before running the training cell.")

NUM_REPLICAS = strategy.num_replicas_in_sync

# ── Seeds ────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"\nSeed        : {SEED} (locked)")

---
## Section 3: Central Configuration

**All key settings are here. Change `TARGET_EPOCHS` to extend training (e.g. 50 → 100) and re-run.**

Settings are kept identical to Vanilla GAN (notebook 01) wherever possible for fair comparison.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  CENTRAL CONFIGURATION — DCGAN RUN 2 (75 EPOCHS)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Dataset ──────────────────────────────────────────────────────────────────
HF_DATASET_NAME        = "evilsocket/alucard-sprites"
IMG_HEIGHT             = 64       # 64x64 RGBA sprites
IMG_WIDTH              = 64
CHANNELS               = 4        # RGBA — preserved alpha channel
PIXEL_NORM             = "tanh"   # normalize to [-1,1]; generator output = tanh
TRAIN_RATIO            = 0.90
VAL_RATIO              = 0.05

# ── Run 2 Architecture Specifications ────────────────────────────────────────
LATENT_DIM             = 100
G_CHANNELS             = [512, 256, 128, 64]   # per upsample stage
D_CHANNELS             = [64, 128, 256, 512]   # per downsample stage
G_INIT_SPATIAL         = 4        # 4x4 initial dense projection
UPSAMPLE_MODE          = "nearest" # 'nearest' eliminates checkerboard overlap
USE_WEIGHT_INIT        = True     # Normal init (mean=0.0, std=0.02)

# ── Run 2 Stabilization Hyperparameters ──────────────────────────────────────
LABEL_SMOOTH_REAL      = 0.9      # One-sided label smoothing for real images
LABEL_FAKE             = 0.0      # Hard fake labels
ENABLE_INSTANCE_NOISE  = True     # Adds small annealed Gaussian noise to D inputs
INSTANCE_NOISE_STD     = 0.05     # Initial noise stddev
NOISE_ANNEAL_EPOCHS    = 20       # Linearly anneal noise stddev to 0.0 by epoch 20

# ── Optimization ─────────────────────────────────────────────────────────────
G_LR                   = 2e-4
D_LR                   = 2e-4
ADAM_BETA1             = 0.5
ADAM_BETA2             = 0.999

# ── Training Run Definition ──────────────────────────────────────────────────
RUN_LABEL              = "DCGAN_Run2_Full_75Epochs"
TARGET_EPOCHS          = 75       # Target epochs for this run
BATCH_SIZE_PER_REPLICA = 64
GLOBAL_BATCH_SIZE      = BATCH_SIZE_PER_REPLICA * NUM_REPLICAS
RESUME_MODE            = "fresh"  # 'fresh' guarantees no accidental Run 1 loading

# ── Execution Staging Mode ───────────────────────────────────────────────────
# Set SMOKE_TEST_MODE = True to run a fast 1,000-image sanity check before full training
SMOKE_TEST_MODE        = False
SMOKE_TEST_SUBSET_SIZE = 1000

# ── Mixed Precision ──────────────────────────────────────────────────────────
USE_MIXED_PRECISION    = False
if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision: float16 enabled")

# ── Logging and Evaluation Intervals ─────────────────────────────────────────
CHECKPOINT_EVERY       = 1        # save resumable checkpoint every N epochs
SAMPLE_EVERY           = 5        # generate sample grid every N epochs
N_FIXED_SAMPLES        = 25       # 5x5 fixed latent grid for temporal comparison
N_EVAL_SAMPLES         = 100      # diversity evaluation
LOG_EVERY              = 1
RESUME_ARCHIVE_PATH    = ""       # Fresh Run 2: empty path ensures fresh init

# ── Paths (Dedicated Run 2 Working Tree) ─────────────────────────────────────
BASE_DIR               = "/kaggle/working/dcgan_280k_run2_75ep"
CKPT_DIR               = f"{BASE_DIR}/checkpoints"
LOG_DIR                = f"{BASE_DIR}/logs"
SAMPLE_DIR             = f"{BASE_DIR}/samples"
CONFIG_DIR             = f"{BASE_DIR}/config"
WEIGHTS_DIR            = f"{BASE_DIR}/weights"
EXPORT_DIR             = f"{BASE_DIR}/export"
ZIP_PATH               = "/kaggle/working/DCGAN_280K_RUN2_75EPOCHS_COMPLETE.zip"

for d in [CKPT_DIR, LOG_DIR, SAMPLE_DIR, CONFIG_DIR, WEIGHTS_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

assert CHANNELS in (3, 4)
assert PIXEL_NORM in ("tanh", "sigmoid")
assert LATENT_DIM > 0
assert TARGET_EPOCHS > 0
assert G_INIT_SPATIAL * (2 ** len(G_CHANNELS)) == IMG_HEIGHT

CFG = dict(
    model="DCGAN_Run2", run_label=RUN_LABEL, hf_dataset=HF_DATASET_NAME,
    img_height=IMG_HEIGHT, img_width=IMG_WIDTH, channels=CHANNELS,
    pixel_norm=PIXEL_NORM, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
    latent_dim=LATENT_DIM, g_channels=G_CHANNELS, d_channels=D_CHANNELS,
    g_init_spatial=G_INIT_SPATIAL, upsample_mode=UPSAMPLE_MODE,
    label_smooth_real=LABEL_SMOOTH_REAL, label_fake=LABEL_FAKE,
    enable_instance_noise=ENABLE_INSTANCE_NOISE, instance_noise_std=INSTANCE_NOISE_STD,
    noise_anneal_epochs=NOISE_ANNEAL_EPOCHS,
    g_lr=G_LR, d_lr=D_LR, beta1=ADAM_BETA1, beta2=ADAM_BETA2,
    target_epochs=TARGET_EPOCHS, batch_size_per_replica=BATCH_SIZE_PER_REPLICA,
    global_batch_size=GLOBAL_BATCH_SIZE, seed=SEED,
    checkpoint_every=CHECKPOINT_EVERY, sample_every=SAMPLE_EVERY, log_every=LOG_EVERY,
    n_fixed_samples=N_FIXED_SAMPLES, n_eval_samples=N_EVAL_SAMPLES, resume_mode=RESUME_MODE,
    base_dir=BASE_DIR, created_at=datetime.utcnow().isoformat()
)
print("DCGAN Run 2 configuration prepared.")
print(f"  Run Label   : {RUN_LABEL}")
print(f"  Target Epochs: {TARGET_EPOCHS}")
print(f"  Upsampling  : UpSampling2D({UPSAMPLE_MODE}) + Conv2D (Anti-Checkerboard)")
print(f"  Real Label  : {LABEL_SMOOTH_REAL} (One-Sided Smoothing)")
print(f"  Base Dir    : {BASE_DIR}")


---
## Section 4: Dataset Discovery

In [ ]:
# ── Install datasets library if needed ───────────────────────────────────────
try:
    from datasets import load_dataset as hf_load
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset as hf_load

print(f"Loading {HF_DATASET_NAME} ...")
raw_dataset = hf_load(HF_DATASET_NAME)
raw_train   = raw_dataset["train"]
print(f"Raw dataset size: {len(raw_train):,} images")

# ── Inspect a sample image ───────────────────────────────────────────────────
sample_item = raw_train[0]
sample_img  = sample_item["image"]
print(f"Sample image mode : {sample_img.mode}")
print(f"Sample image size : {sample_img.size}  (W x H)")
sample_arr = np.array(sample_img)
print(f"Sample array shape: {sample_arr.shape}")
print(f"Sample dtype      : {sample_arr.dtype}")
print(f"Sample value range: [{sample_arr.min()}, {sample_arr.max()}]")

# ── Deduplication (load cached indices if available) ─────────────────────────
UNIQUE_IDX_PATH   = f"{CONFIG_DIR}/unique_indices.json"
SHUFFLED_IDX_PATH = f"{CONFIG_DIR}/shuffled_indices.json"
SPLIT_META_PATH   = f"{CONFIG_DIR}/split_metadata.json"

if os.path.exists(UNIQUE_IDX_PATH):
    print("\nLoading cached unique indices ...")
    with open(UNIQUE_IDX_PATH) as f:
        unique_indices = json.load(f)
    print(f"Unique images (cached): {len(unique_indices):,}")
else:
    print("\nDeduplicating by SHA-256 pixel hash ...")
    seen, unique_indices = set(), []
    rejected = 0
    for i, item in enumerate(raw_train):
        if i % 10000 == 0:
            print(f"  {i:,}/{len(raw_train):,}  unique so far: {len(unique_indices):,}")
        try:
            arr = np.asarray(item["image"].convert("RGBA"), dtype=np.uint8)
            if arr.shape != (128, 128, 4):
                rejected += 1; continue
            h = hashlib.sha256(arr.tobytes()).hexdigest()
            if h not in seen:
                seen.add(h)
                unique_indices.append(i)
        except Exception:
            rejected += 1
    with open(UNIQUE_IDX_PATH, "w") as f:
        json.dump(unique_indices, f)
    print(f"Deduplication complete: {len(unique_indices):,} unique, {rejected} rejected")

unique_dataset = raw_train.select(unique_indices)
print(f"\nDataset ready: {len(unique_dataset):,} unique sprites")

---
## Section 5: Preprocessing and tf.data Pipeline

In [ ]:
# ── Normalization helpers ─────────────────────────────────────────────────────
def normalize(arr_float32):
    """[0,1] float32 -> [-1,1] for tanh."""
    if PIXEL_NORM == "tanh":
        return arr_float32 * 2.0 - 1.0
    return arr_float32

def denormalize(arr):
    """Invert normalize() for display."""
    if PIXEL_NORM == "tanh":
        return np.clip((arr + 1.0) / 2.0, 0.0, 1.0)
    return np.clip(arr, 0.0, 1.0)

def preprocess_image(item):
    """PIL image -> normalized float32 numpy array at training resolution."""
    from PIL import Image as PILImage
    img = item["image"].convert("RGBA")
    if img.size != (IMG_WIDTH, IMG_HEIGHT):
        img = img.resize((IMG_WIDTH, IMG_HEIGHT), PILImage.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    return normalize(arr)

def make_generator_fn(hf_ds):
    def _gen():
        for item in hf_ds:
            yield preprocess_image(item)
    return _gen

def build_tf_dataset(hf_ds, batch_size, shuffle=True, repeat=True):
    ds = tf.data.Dataset.from_generator(
        make_generator_fn(hf_ds),
        output_signature=tf.TensorSpec(shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), dtype=tf.float32)
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=min(5000, len(hf_ds)), seed=SEED)
    ds = ds.batch(batch_size, drop_remainder=True)
    if repeat:
        ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)

print(f"Preprocessing config:")
print(f"  Resolution    : {IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}")
print(f"  Normalization : {PIXEL_NORM} -> {'[-1,1]' if PIXEL_NORM=='tanh' else '[0,1]'}")
print(f"  Channel policy: RGBA preserved (alpha channel included in generation target)")
print("Pipeline functions defined. Datasets will be built after split in Section 7.")

---
## Section 6: Dataset Previews

In [ ]:
def show_sprite_grid(hf_ds, n=32, title="Sprites", save_path=None):
    cols = 8; rows = max(1, n // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.6, rows*1.6))
    axes = axes.flatten()
    indices = random.sample(range(len(hf_ds)), min(n, len(hf_ds)))
    for ax, idx in zip(axes, indices):
        arr = np.array(hf_ds[idx]["image"].convert("RGBA"), dtype=np.float32) / 255.0
        ax.imshow(arr); ax.axis("off")
    for ax in axes[len(indices):]: ax.axis("off")
    plt.suptitle(title, fontsize=12); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show(); plt.close()

show_sprite_grid(unique_dataset, n=32, title="Original Sprites (128x128 RGBA)",
                 save_path=f"{SAMPLE_DIR}/original_preview.png")
print("Preview saved.")

---
## Section 7: Train / Validation / Test Split

The split convention matches Vanilla GAN: 90% train, 5% validation, 5% test, seed 42. Both notebooks deduplicate in source order and shuffle the resulting unique-image indices deterministically. For exact identity, provide the Vanilla GAN `shuffled_indices.json` through `SHARED_SPLIT_MANIFEST` if available; otherwise the DCGAN notebook reproduces the same split only if dataset ordering and deduplication are identical. The notebook saves its manifest and metadata for later comparisons.

In [ ]:
# Deterministic split compatible with the Vanilla GAN notebook.
# Both notebooks deduplicate in original dataset order, then shuffle the unique-index list with SEED.
# If a shared split manifest is explicitly mounted, use it; otherwise reproduce the same split locally.
SHARED_SPLIT_MANIFEST = ""  # Optional path to Vanilla GAN's shuffled_indices.json
if SHARED_SPLIT_MANIFEST and os.path.isfile(SHARED_SPLIT_MANIFEST):
    print(f"Loading shared split manifest: {SHARED_SPLIT_MANIFEST}")
    with open(SHARED_SPLIT_MANIFEST) as f:
        shuffled = json.load(f)
    if len(shuffled) != len(unique_dataset):
        raise ValueError("Shared split manifest length does not match this deduplicated dataset.")
else:
    if os.path.exists(SHUFFLED_IDX_PATH) and os.path.exists(SPLIT_META_PATH):
        print("Loading cached deterministic split ...")
        with open(SHUFFLED_IDX_PATH) as f:
            shuffled = json.load(f)
    else:
        print(f"Creating deterministic split (seed={SEED}) ...")
        shuffled = list(range(len(unique_dataset)))
        random.Random(SEED).shuffle(shuffled)
        with open(SHUFFLED_IDX_PATH, "w") as f:
            json.dump(shuffled, f)

TOTAL = len(shuffled)
TRAIN_SIZE = int(round(TOTAL * TRAIN_RATIO))
VAL_SIZE = int(round(TOTAL * VAL_RATIO))
TEST_SIZE = TOTAL - TRAIN_SIZE - VAL_SIZE
split_meta = dict(total=TOTAL, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE,
                  seed=SEED, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                  img_height=IMG_HEIGHT, img_width=IMG_WIDTH, channels=CHANNELS,
                  pixel_norm=PIXEL_NORM, manifest_source=(SHARED_SPLIT_MANIFEST or SHUFFLED_IDX_PATH),
                  created_at=datetime.utcnow().isoformat())
with open(SHUFFLED_IDX_PATH, "w") as f:
    json.dump(shuffled, f)
with open(SPLIT_META_PATH, "w") as f:
    json.dump(split_meta, f, indent=2)

train_ids = shuffled[:TRAIN_SIZE]
val_ids = shuffled[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]
test_ids = shuffled[TRAIN_SIZE + VAL_SIZE:]
assert not (set(train_ids) & set(val_ids))
assert not (set(train_ids) & set(test_ids))
assert not (set(val_ids) & set(test_ids))

hf_train = unique_dataset.select(train_ids)
hf_val = unique_dataset.select(val_ids)
hf_test = unique_dataset.select(test_ids)

# Build ordinary batched datasets with GLOBAL batch size, then distribute exactly once.
train_ds_unsharded = build_tf_dataset(hf_train, GLOBAL_BATCH_SIZE, shuffle=True, repeat=True)
val_ds_unsharded = build_tf_dataset(hf_val, GLOBAL_BATCH_SIZE, shuffle=False, repeat=False)
train_ds = strategy.experimental_distribute_dataset(train_ds_unsharded)
val_ds = strategy.experimental_distribute_dataset(val_ds_unsharded)
STEPS_PER_EPOCH = TRAIN_SIZE // GLOBAL_BATCH_SIZE
if STEPS_PER_EPOCH < 1:
    raise ValueError("Training split is smaller than one global batch.")

print("Split verified: no overlap")
print(f"Train: {TRAIN_SIZE:,} | Val: {VAL_SIZE:,} | Test: {TEST_SIZE:,}")
print(f"Global batch: {GLOBAL_BATCH_SIZE} | per replica: {BATCH_SIZE_PER_REPLICA} | replicas: {NUM_REPLICAS}")
print(f"Steps/epoch: {STEPS_PER_EPOCH}")
print("Note: identical split indices are reproducible when dataset ordering and deduplication match Vanilla GAN.")

---
## Section 8: DCGAN Architecture

In [ ]:
def dcgan_weight_init(model):
    for layer in model.layers:
        if hasattr(layer, "kernel") and layer.kernel is not None:
            layer.kernel.assign(tf.random.normal(layer.kernel.shape, 0.0, 0.02))
        if hasattr(layer, "bias") and layer.bias is not None:
            layer.bias.assign(tf.zeros_like(layer.bias))
        if isinstance(layer, layers.BatchNormalization):
            if layer.gamma is not None:
                layer.gamma.assign(tf.random.normal(layer.gamma.shape, 1.0, 0.02))
            if layer.beta is not None:
                layer.beta.assign(tf.zeros_like(layer.beta))

def build_generator(latent_dim, g_channels, init_spatial, img_h, img_w, channels, pixel_norm, upsample_mode='nearest'):
    """
    DCGAN Run 2 Generator: Nearest-neighbor upsampling + 3x3 Conv2D.
    Eliminates stride-2 transposed convolution checkerboard artifacts.
    """
    inp = keras.Input(shape=(latent_dim,), name="z")
    x = layers.Dense(init_spatial * init_spatial * g_channels[0], use_bias=False)(inp)
    x = layers.Reshape((init_spatial, init_spatial, g_channels[0]))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for ch in g_channels[1:]:
        x = layers.UpSampling2D(size=(2, 2), interpolation=upsample_mode)(x)
        x = layers.Conv2D(ch, kernel_size=3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

    act = "tanh" if pixel_norm == "tanh" else "sigmoid"
    x = layers.UpSampling2D(size=(2, 2), interpolation=upsample_mode)(x)
    x = layers.Conv2D(channels, kernel_size=3, padding="same", activation=act, use_bias=True, name="out")(x)
    return keras.Model(inp, x, name="generator_run2")

def build_discriminator(img_h, img_w, channels, d_channels):
    """
    DCGAN Run 2 Discriminator: No BatchNormalization in layer 1, LeakyReLU(0.2).
    Returns scalar logit for BCE loss.
    """
    inp = keras.Input(shape=(img_h, img_w, channels), name="image")
    # Layer 1: No BatchNormalization per DCGAN standard guidelines
    x = layers.Conv2D(d_channels[0], kernel_size=4, strides=2, padding="same", use_bias=False)(inp)
    x = layers.LeakyReLU(0.2)(x)
    for ch in d_channels[1:]:
        x = layers.Conv2D(ch, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1, name="logit")(x)
    return keras.Model(inp, x, name="discriminator_run2")

with strategy.scope():
    generator     = build_generator(LATENT_DIM, G_CHANNELS, G_INIT_SPATIAL, IMG_HEIGHT, IMG_WIDTH, CHANNELS, PIXEL_NORM, UPSAMPLE_MODE)
    discriminator = build_discriminator(IMG_HEIGHT, IMG_WIDTH, CHANNELS, D_CHANNELS)
    g_optimizer   = keras.optimizers.Adam(G_LR, beta_1=ADAM_BETA1, beta_2=ADAM_BETA2)
    d_optimizer   = keras.optimizers.Adam(D_LR, beta_1=ADAM_BETA1, beta_2=ADAM_BETA2)

if USE_WEIGHT_INIT:
    dcgan_weight_init(generator)
    dcgan_weight_init(discriminator)
    print("DCGAN Run 2 normal weight init applied.")

generator.summary()
discriminator.summary()
print(f"G params: {generator.count_params():,} | D params: {discriminator.count_params():,}")

_z = tf.random.normal((2, LATENT_DIM))
_img = generator(_z, training=False)
_log = discriminator(_img, training=False)
assert _img.shape == (2, IMG_HEIGHT, IMG_WIDTH, CHANNELS), f"Unexpected G output shape: {_img.shape}"
assert _log.shape == (2, 1), f"Unexpected D output shape: {_log.shape}"
print("DCGAN Run 2 Forward-pass smoke test: PASSED")


---
## Section 9: Losses and Training Step

In [ ]:
# DCGAN Run 2 Losses, One-Sided Label Smoothing, Instance Noise, and Distributed Training Step
_bce_none = keras.losses.BinaryCrossentropy(from_logits=True, reduction=keras.losses.Reduction.NONE)

def _per_example_bce(labels, logits):
    vals = _bce_none(labels, logits)
    return tf.reshape(vals, [-1])

def d_loss_fn(real_logits, fake_logits, label_smooth_real=0.9):
    # Real label smoothing: target is 0.9 rather than 1.0 to prevent D logit saturation
    real_targets = tf.fill(tf.shape(real_logits), tf.cast(label_smooth_real, real_logits.dtype))
    fake_targets = tf.zeros_like(fake_logits)
    real_per = _per_example_bce(real_targets, real_logits)
    fake_per = _per_example_bce(fake_targets, fake_logits)
    return (tf.nn.compute_average_loss(real_per, global_batch_size=GLOBAL_BATCH_SIZE) +
            tf.nn.compute_average_loss(fake_per, global_batch_size=GLOBAL_BATCH_SIZE))

def g_loss_fn(fake_logits):
    # Generator always trains toward 1.0
    fake_per = _per_example_bce(tf.ones_like(fake_logits), fake_logits)
    return tf.nn.compute_average_loss(fake_per, global_batch_size=GLOBAL_BATCH_SIZE)

# Diagnostic probability metric helper (logits -> sigmoid for logging without altering loss)
def logits_to_prob(logits):
    return tf.reduce_mean(tf.sigmoid(logits))

@tf.function
def train_step_per_replica(real_images, current_noise_std):
    local_batch = tf.shape(real_images)[0]
    z = tf.random.normal((local_batch, LATENT_DIM))
    
    # Optional instance noise injection to smooth discriminator decision boundaries
    if current_noise_std > 0.0:
        real_in = real_images + tf.random.normal(tf.shape(real_images), 0.0, current_noise_std)
    else:
        real_in = real_images

    with tf.GradientTape() as d_tape:
        fake_images = generator(z, training=True)
        fake_in = fake_images + tf.random.normal(tf.shape(fake_images), 0.0, current_noise_std) if current_noise_std > 0.0 else fake_images
        real_logits = discriminator(real_in, training=True)
        fake_logits = discriminator(fake_in, training=True)
        d_loss = d_loss_fn(real_logits, fake_logits, label_smooth_real=LABEL_SMOOTH_REAL)
    
    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    d_pairs = [(g, v) for g, v in zip(d_grads, discriminator.trainable_variables) if g is not None]
    d_optimizer.apply_gradients(d_pairs)

    with tf.GradientTape() as g_tape:
        fake_images_g = generator(z, training=True)
        fake_logits_g = discriminator(fake_images_g, training=True)
        g_loss = g_loss_fn(fake_logits_g)
        
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    g_pairs = [(g, v) for g, v in zip(g_grads, generator.trainable_variables) if g is not None]
    g_optimizer.apply_gradients(g_pairs)

    d_real_prob = logits_to_prob(real_logits)
    d_fake_prob = logits_to_prob(fake_logits)
    return g_loss, d_loss, d_real_prob, d_fake_prob

@tf.function
def distributed_train_step(distributed_real_images, current_noise_std):
    per_g, per_d, per_d_real, per_d_fake = strategy.run(train_step_per_replica, args=(distributed_real_images, current_noise_std))
    g_loss = strategy.reduce(tf.distribute.ReduceOp.SUM, per_g, axis=None)
    d_loss = strategy.reduce(tf.distribute.ReduceOp.SUM, per_d, axis=None)
    d_real_prob = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_d_real, axis=None)
    d_fake_prob = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_d_fake, axis=None)
    return g_loss, d_loss, d_real_prob, d_fake_prob

print("DCGAN Run 2 losses and distributed step defined with label smoothing and instance noise.")


---
## Section 10: Checkpoint and Resume Infrastructure

In [ ]:
# Full-state checkpointing and resume infrastructure for DCGAN Run 2.
ckpt = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                           g_optimizer=g_optimizer, d_optimizer=d_optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, directory=CKPT_DIR,
                                         max_to_keep=5, checkpoint_name="ckpt")
STATE_PATH = f"{CONFIG_DIR}/training_state.json"
HISTORY_CSV = f"{LOG_DIR}/training_history.csv"
HISTORY_JSON = f"{LOG_DIR}/training_history.json"
RUN_CONFIG_PATH = f"{CONFIG_DIR}/run_config.json"

# Save current effective configuration
with open(RUN_CONFIG_PATH, "w") as f:
    json.dump(CFG, f, indent=2)

def save_state(epoch, global_step, history, best_epoch, best_g_loss):
    state = dict(epoch=int(epoch), global_step=int(global_step), best_epoch=int(best_epoch),
                 best_g_loss=float(best_g_loss), target_epochs=int(TARGET_EPOCHS),
                 model="DCGAN_Run2", run_label=RUN_LABEL, img_height=IMG_HEIGHT, img_width=IMG_WIDTH,
                 channels=CHANNELS, latent_dim=LATENT_DIM, g_channels=G_CHANNELS, d_channels=D_CHANNELS,
                 seed=SEED, global_batch_size=GLOBAL_BATCH_SIZE,
                 saved_at=datetime.utcnow().isoformat())
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, STATE_PATH)
    tmp_hist = HISTORY_JSON + ".tmp"
    with open(tmp_hist, "w") as f:
        json.dump(history, f, indent=2)
    os.replace(tmp_hist, HISTORY_JSON)

def load_state():
    if os.path.isfile(STATE_PATH):
        with open(STATE_PATH) as f:
            return json.load(f)
    return None

def load_history():
    if os.path.isfile(HISTORY_JSON):
        with open(HISTORY_JSON) as f:
            return json.load(f)
    return {"epoch": [], "g_loss": [], "d_loss": [], "d_real_prob": [], "d_fake_prob": [], "epoch_time": [], "global_step": []}

start_epoch = 0
global_step = 0
best_g_loss = float("inf")
best_epoch = -1
history = load_history()
latest = ckpt_manager.latest_checkpoint

# Fresh Run 2 Initialization
if RESUME_MODE == "fresh" or not latest:
    print("[RUN 2 INITIALIZATION] Fresh start initialized at epoch 0 with new weights.")
elif RESUME_MODE in ("auto", "required") and latest:
    status = ckpt.restore(latest)
    status.expect_partial()
    st = load_state()
    if st and st.get("model") == "DCGAN_Run2":
        start_epoch = int(st["epoch"]) + 1
        global_step = int(st["global_step"])
        best_g_loss = float(st["best_g_loss"])
        best_epoch = int(st["best_epoch"])
        print(f"[RUN 2 RESUME] Resumed Run 2 after epoch {start_epoch}; step {global_step}")
    else:
        raise ValueError("Incompatible checkpoint state; refusing to restore non-Run 2 weights.")

print(f"Run 2 will train epochs {start_epoch} through {TARGET_EPOCHS - 1} (target total = {TARGET_EPOCHS}).")


---
## Section 11: Checkpoint Smoke Test

In [ ]:
print("--- DCGAN checkpoint smoke test ---")
_smoke_prefix = f"{CKPT_DIR}/smoke_test"
_smoke_ckpt = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                                  g_optimizer=g_optimizer, d_optimizer=d_optimizer,
                                  smoke_step=tf.Variable(7, dtype=tf.int64))
_smoke_var = generator.trainable_variables[0]
_before = _smoke_var.numpy().copy()
_smoke_ckpt.write(_smoke_prefix)
_smoke_var.assign(tf.zeros_like(_smoke_var))
_smoke_ckpt.smoke_step.assign(0)
_smoke_ckpt.read(_smoke_prefix).expect_partial()
assert np.allclose(_before, _smoke_var.numpy()), "Generator weights did not restore"
assert int(_smoke_ckpt.smoke_step.numpy()) == 7, "Checkpoint metadata did not restore"
for _f in Path(CKPT_DIR).glob("smoke_test*"):
    try: _f.unlink()
    except OSError: pass
print("Checkpoint write/read smoke test: PASSED")
print("Note: full fresh-process optimizer-slot restoration must also be validated on Kaggle before relying on recovery.")

---
## Section 12: Fixed Latent Vectors and Initial Sample Grid

In [ ]:
FIXED_Z_PATH = f"{CONFIG_DIR}/fixed_latent_vectors.npy"
if os.path.exists(FIXED_Z_PATH):
    fixed_z = np.load(FIXED_Z_PATH)
    print(f"Loaded fixed_z: {fixed_z.shape}")
else:
    fixed_z = np.random.RandomState(SEED).randn(N_FIXED_SAMPLES, LATENT_DIM).astype(np.float32)
    np.save(FIXED_Z_PATH, fixed_z)
    print(f"Created fixed_z: {fixed_z.shape}")

def save_sample_grid(label, path, use_fixed=True):
    z = fixed_z if use_fixed else np.random.randn(N_FIXED_SAMPLES, LATENT_DIM).astype(np.float32)
    imgs = denormalize(generator(z, training=False).numpy())
    n = len(imgs); cols = 5; rows = max(1, n // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 1.8))
    axes = axes.flatten()
    for ax, img in zip(axes, imgs):
        ax.imshow(np.clip(img, 0, 1))
        ax.axis("off")
    for ax in axes[n:]: ax.axis("off")
    plt.suptitle(f"DCGAN Run 2 — {label}", fontsize=11)
    plt.tight_layout()
    plt.savefig(path, dpi=100, bbox_inches="tight")
    plt.close()

save_sample_grid("Epoch 0 (Initial Untrained Run 2 Model)", f"{SAMPLE_DIR}/samples_epoch_0000.png")
print("Initial sample grid saved at:", f"{SAMPLE_DIR}/samples_epoch_0000.png")


---
## Section 13: Training Loop

In [ ]:
if start_epoch >= TARGET_EPOCHS:
    print(f"Nothing to train: {start_epoch} completed epochs >= target {TARGET_EPOCHS}.")
else:
    if not gpus:
        print("WARNING: No GPU detected. Running on CPU may be slow.")
    print(f"Training DCGAN Run 2 epochs {start_epoch + 1} to {TARGET_EPOCHS} | steps/epoch={STEPS_PER_EPOCH}")
    train_iter = iter(train_ds)
    for epoch in range(start_epoch, TARGET_EPOCHS):
        t0 = time.time()
        g_losses, d_losses, d_reals, d_fakes = [], [], [], []
        
        # Calculate current instance noise level (linear decay to 0.0 by NOISE_ANNEAL_EPOCHS)
        if ENABLE_INSTANCE_NOISE and epoch < NOISE_ANNEAL_EPOCHS:
            current_noise = INSTANCE_NOISE_STD * (1.0 - (epoch / float(NOISE_ANNEAL_EPOCHS)))
        else:
            current_noise = 0.0
            
        try:
            for _ in range(STEPS_PER_EPOCH):
                g_l, d_l, p_r, p_f = distributed_train_step(next(train_iter), current_noise)
                g_val, d_val = float(g_l.numpy()), float(d_l.numpy())
                if not (np.isfinite(g_val) and np.isfinite(d_val)):
                    raise FloatingPointError(f"Non-finite loss at epoch {epoch + 1}, step {global_step}")
                g_losses.append(g_val)
                d_losses.append(d_val)
                d_reals.append(float(p_r.numpy()))
                d_fakes.append(float(p_f.numpy()))
                global_step += 1
        except KeyboardInterrupt:
            print("Training interrupted. State preserved up to last epoch checkpoint.")
            raise

        mg = float(np.mean(g_losses))
        md = float(np.mean(d_losses))
        m_real = float(np.mean(d_reals))
        m_fake = float(np.mean(d_fakes))
        elapsed = time.time() - t0

        history["epoch"].append(epoch)
        history["g_loss"].append(mg)
        history["d_loss"].append(md)
        history["d_real_prob"].append(m_real)
        history["d_fake_prob"].append(m_fake)
        history["epoch_time"].append(elapsed)
        history["global_step"].append(global_step)

        if mg < best_g_loss:
            best_g_loss, best_epoch = mg, epoch
            generator.save_weights(f"{WEIGHTS_DIR}/generator_best.weights.h5")
            discriminator.save_weights(f"{WEIGHTS_DIR}/discriminator_best.weights.h5")

        # Save history atomically
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(HISTORY_CSV, index=False)
        with open(HISTORY_JSON + ".tmp", "w") as f:
            json.dump(history, f, indent=2)
        os.replace(HISTORY_JSON + ".tmp", HISTORY_JSON)

        # Save periodic checkpoint
        if (epoch + 1) % CHECKPOINT_EVERY == 0 or epoch == TARGET_EPOCHS - 1:
            ckpt_manager.save(checkpoint_number=epoch)
            save_state(epoch, global_step, history, best_epoch, best_g_loss)

        # Save visual sample grid
        if (epoch + 1) % SAMPLE_EVERY == 0 or epoch == TARGET_EPOCHS - 1:
            save_sample_grid(f"Epoch {epoch + 1}", f"{SAMPLE_DIR}/samples_epoch_{epoch + 1:04d}.png", use_fixed=True)
            save_sample_grid(f"Epoch {epoch + 1} (Fresh Noise)", f"{SAMPLE_DIR}/fresh_samples_epoch_{epoch + 1:04d}.png", use_fixed=False)

        print(f"Epoch {epoch + 1:3d}/{TARGET_EPOCHS} | G:{mg:.4f} D:{md:.4f} | D(real):{m_real:.3f} D(fake):{m_fake:.3f} | Noise:{current_noise:.3f} | {elapsed:.1f}s | step={global_step}")

    print(f"Training loop completed: {len(history['epoch'])} epochs. Best G loss={best_g_loss:.4f} at epoch {best_epoch + 1 if best_epoch >= 0 else 'N/A'}")


---
## Section 14: Monitoring — Loss Curves

In [ ]:
hist_df = pd.read_csv(HISTORY_CSV)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1. G & D Losses
axes[0].plot(hist_df["epoch"] + 1, hist_df["g_loss"], label="G Loss", color="#3b82f6", lw=1.5)
axes[0].plot(hist_df["epoch"] + 1, hist_df["d_loss"], label="D Loss", color="#ef4444", lw=1.5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("DCGAN Run 2 Training Losses")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Discriminator Probabilities D(real) and D(fake)
if "d_real_prob" in hist_df.columns:
    axes[1].plot(hist_df["epoch"] + 1, hist_df["d_real_prob"], label="D(real)", color="#10b981", lw=1.5)
    axes[1].plot(hist_df["epoch"] + 1, hist_df["d_fake_prob"], label="D(fake)", color="#f59e0b", lw=1.5)
    axes[1].axhline(0.5, color="gray", linestyle="--", alpha=0.5)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Mean Output Probability")
    axes[1].set_title("Discriminator Equilibrium Diagnostics")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# 3. Epoch Duration
axes[2].plot(hist_df["epoch"] + 1, hist_df["epoch_time"], color="#8b5cf6", lw=1.5)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Seconds")
axes[2].set_title("Epoch Duration")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{LOG_DIR}/loss_curves.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close()
print("DCGAN Run 2 loss and diagnostic curves saved.")


---
## Section 15: Final Model Selection

In [ ]:
generator.save_weights(f"{WEIGHTS_DIR}/generator_latest.weights.h5")
discriminator.save_weights(f"{WEIGHTS_DIR}/discriminator_latest.weights.h5")
generator.save(f"{WEIGHTS_DIR}/generator_latest.keras")
discriminator.save(f"{WEIGHTS_DIR}/discriminator_latest.keras")
print("Latest weights saved.")
print(f"Best checkpoint: epoch {best_epoch+1}, G loss {best_g_loss:.4f}")
print("NOTE: Inspect sample grids to visually confirm best epoch.")

---
## Section 16: Evaluation

In [ ]:
import hashlib as _hl
rng_eval  = np.random.RandomState(SEED+1)
eval_z    = rng_eval.randn(N_EVAL_SAMPLES,LATENT_DIM).astype(np.float32)
eval_imgs = denormalize(generator(eval_z,training=False).numpy())

flat_eval     = eval_imgs.reshape(N_EVAL_SAMPLES,-1)
diversity_std = float(flat_eval.std(axis=0).mean())
print(f"Diversity (mean pixel std): {diversity_std:.4f}")
if diversity_std<0.01: print("[WARN] Very low diversity — possible mode collapse.")

gen_hashes   = [_hl.sha256((img*255).astype("uint8").tobytes()).hexdigest() for img in eval_imgs]
n_unique_gen = len(set(gen_hashes))
print(f"Unique generated: {n_unique_gen}/{N_EVAL_SAMPLES}")

print("Computing NN distances vs 200 test sprites ...")
test_items = [hf_test[i] for i in range(min(200,len(hf_test)))]
test_arrs  = np.stack([denormalize(preprocess_image(it)) for it in test_items])
n_cmp      = min(len(eval_imgs), len(test_arrs))
gen_flat   = eval_imgs[:n_cmp].reshape(n_cmp, -1)
test_flat  = test_arrs[:n_cmp].reshape(n_cmp, -1)
nn_dists   = np.linalg.norm(gen_flat[:,None,:]-test_flat[None,:,:],axis=-1).min(axis=1)
print(f"Mean NN dist (L2): {nn_dists.mean():.4f}  std: {nn_dists.std():.4f}")

eval_report = dict(model="DCGAN",epochs_completed=len(history["epoch"]),
    best_epoch=best_epoch,best_g_loss=best_g_loss,
    diversity_std=diversity_std,unique_generated=n_unique_gen,total_generated=N_EVAL_SAMPLES,
    nn_mean_dist=float(nn_dists.mean()),nn_std_dist=float(nn_dists.std()),
    latent_dim=LATENT_DIM,img_size=f"{IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}",
    g_channels=G_CHANNELS,d_channels=D_CHANNELS)
with open(f"{LOG_DIR}/evaluation_report.json","w") as f: json.dump(eval_report,f,indent=2)
print("Evaluation report saved.")

---
## Section 17: Artifact Export

In [ ]:
def export_all_artifacts():
    """Export current state without fabricating a completed epoch when training has not started."""
    last_completed = history["epoch"][-1] if history["epoch"] else None
    if last_completed is not None:
        ckpt_manager.save(checkpoint_number=int(last_completed))
        save_state(last_completed, global_step, history, best_epoch, best_g_loss)
    generator.save_weights(f"{WEIGHTS_DIR}/generator_latest.weights.h5")
    discriminator.save_weights(f"{WEIGHTS_DIR}/discriminator_latest.weights.h5")
    generator.save(f"{WEIGHTS_DIR}/generator_latest.keras")
    discriminator.save(f"{WEIGHTS_DIR}/discriminator_latest.keras")
    label = f"Epoch {last_completed + 1}" if last_completed is not None else "Initial untrained model"
    save_sample_grid(label, f"{SAMPLE_DIR}/samples_final.png")

    import io as _io
    for model, name in [(generator, "generator"), (discriminator, "discriminator")]:
        buf = _io.StringIO()
        model.summary(print_fn=lambda line: buf.write(line + "\n"))
        with open(f"{CONFIG_DIR}/{name}_summary.txt", "w") as f:
            f.write(buf.getvalue())

    ds_info = dict(source=HF_DATASET_NAME, unique_images=len(unique_dataset),
                   train=TRAIN_SIZE, val=VAL_SIZE, test=TEST_SIZE,
                   img_height=IMG_HEIGHT, img_width=IMG_WIDTH, channels=CHANNELS,
                   pixel_norm=PIXEL_NORM, seed=SEED,
                   train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                   global_batch_size=GLOBAL_BATCH_SIZE, batch_size_per_replica=BATCH_SIZE_PER_REPLICA)
    with open(f"{CONFIG_DIR}/dataset_info.json", "w") as f:
        json.dump(ds_info, f, indent=2)

    # Explicitly preserve the deterministic index manifest and split metadata in the archive.
    for source, target_name in [(SHUFFLED_IDX_PATH, "shuffled_indices.json"),
                                (SPLIT_META_PATH, "split_metadata.json"),
                                (UNIQUE_IDX_PATH, "unique_indices.json")]:
        if os.path.isfile(source):
            target = os.path.join(CONFIG_DIR, target_name)
            if os.path.abspath(source) != os.path.abspath(target):
                shutil.copy2(source, target)

    completed_epochs = (last_completed + 1) if last_completed is not None else 0
    readme = (
        "# DCGAN 280K Artifact Bundle\n\n"
        "This archive contains DCGAN model files, configuration, logs, samples, and resumable TensorFlow checkpoints when training has completed at least one epoch.\n\n"
        "## Resume in a fresh Kaggle session\n"
        "1. Attach this ZIP as a Kaggle input dataset.\n"
        "2. Set RESUME_ARCHIVE_PATH to the mounted ZIP path in Section 3.\n"
        "3. Set TARGET_EPOCHS=100 (total target) and RESUME_MODE='auto' or 'required'.\n"
        "4. Run cells in order. The archive is extracted only when no local checkpoint exists.\n\n"
        f"Completed epochs at export: {completed_epochs}.\n\n"
        "## Important\n"
        "Kaggle /kaggle/working storage is temporary. Download the ZIP or save a versioned Kaggle output before the session ends.\n"
    )
    with open(f"{BASE_DIR}/README.md", "w") as f:
        f.write(readme)
    print(f"Artifact files exported. Completed epochs: {completed_epochs}.")

export_all_artifacts()

---
## Section 18: ZIP Creation and Verification

In [ ]:
import hashlib as _hl

REQUIRED_PATTERNS = [
    "weights/generator_latest.keras",
    "weights/generator_latest.weights.h5",
    "weights/discriminator_latest.keras",
    "weights/discriminator_latest.weights.h5",
    "checkpoints/checkpoint",
    "logs/training_history.csv",
    "logs/training_history.json",
    "config/run_config.json",
    "config/fixed_latent_vectors.npy",
    "config/training_state.json",
    "config/shuffled_indices.json",
    "config/split_metadata.json",
    "samples/samples_final.png",
    "README.md",
]

def _sha256_file(path):
    h = _hl.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def create_and_verify_zip(zip_path, source_dir):
    # Inventory excludes the ZIP itself and its staging file if either lives nearby.
    inv = []
    for root, _, files in os.walk(source_dir):
        for fname in files:
            full = os.path.join(root, fname)
            rel = os.path.relpath(full, source_dir)
            inv.append({"path": rel, "size_bytes": os.path.getsize(full), "sha256": _sha256_file(full)})
    manifest_path = os.path.join(source_dir, "artifact_manifest.json")
    with open(manifest_path + ".tmp", "w") as f:
        json.dump(inv, f, indent=2)
    os.replace(manifest_path + ".tmp", manifest_path)

    staging = zip_path + ".staging"
    try:
        with zipfile.ZipFile(staging, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as zf:
            for root, _, files in os.walk(source_dir):
                for fname in files:
                    full = os.path.join(root, fname)
                    zf.write(full, os.path.relpath(full, source_dir))
        with zipfile.ZipFile(staging, "r") as zf:
            bad = zf.testzip()
            if bad is not None:
                raise RuntimeError(f"ZIP integrity failure at member: {bad}")
            names = set(zf.namelist())
            missing = [p for p in REQUIRED_PATTERNS if not any(n.endswith(p) for n in names)]
            if missing:
                raise RuntimeError(f"ZIP missing required artifacts: {missing}")
        # Only replace the previous archive after the staged archive passes checks.
        os.replace(staging, zip_path)
        size_mb = os.path.getsize(zip_path) / 1e6
        print(f"ZIP verified: {zip_path}")
        print(f"Size: {size_mb:.2f} MB | files: {len(names)} | required artifacts: PASS | CRC test: PASS")
        return zip_path, size_mb
    except Exception:
        if os.path.exists(staging):
            os.remove(staging)
        raise

zip_path, zip_mb = create_and_verify_zip(ZIP_PATH, BASE_DIR)

---
## Section 19: Download and Persistence

In [ ]:
from IPython.display import display, FileLink, Markdown

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"Expected archive not found: {ZIP_PATH}")
print(f"Archive created and integrity-checked: {ZIP_PATH}")
print(f"Archive size: {os.path.getsize(ZIP_PATH) / 1e6:.2f} MB")
print("Download status: the file is available in this notebook session; it is not confirmed downloaded until you click the link.")
display(FileLink(ZIP_PATH, result_html_prefix="Download DCGAN archive: "))
print("For persistence: save/commit a Kaggle notebook version with output files, or download this ZIP before the session ends. /kaggle/working is temporary.")

---
## Section 20: Resume Instructions

In [ ]:
st=load_state()
if st:
    print(f"Last epoch : {st['epoch']+1}")
    print(f"Global step: {st['global_step']}")
    print(f"Best G loss: {st['best_g_loss']:.4f} (epoch {st['best_epoch']+1})")
else:
    print("No state found yet.")

print("""
--- Resume from epoch 50 to 100 ---
1. Download DCGAN_280K_COMPLETE.zip from Output tab.
2. Upload as Kaggle dataset (My Datasets -> New Dataset).
3. In new session, copy files:
     import shutil, glob
     for f in glob.glob("/kaggle/input/<name>/checkpoints/*"):
         shutil.copy(f, "/kaggle/working/dcgan_280k/checkpoints/")
     for f in glob.glob("/kaggle/input/<name>/config/*"):
         shutil.copy(f, "/kaggle/working/dcgan_280k/config/")
4. Set TARGET_EPOCHS=100, RESUME_MODE="auto" in Section 3.
5. Run all cells.
NOTE: TARGET_EPOCHS is TOTAL target, not additional epochs.
""")

---
## Section 21: Final Experiment Report

In [ ]:
hist_df=pd.read_csv(HISTORY_CSV) if os.path.exists(HISTORY_CSV) else pd.DataFrame()

report=dict(
    experiment="DCGAN 280K Alucard Sprites",
    dataset=HF_DATASET_NAME, unique_images=len(unique_dataset),
    train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE,
    img_resolution=f"{IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}",
    pixel_norm=PIXEL_NORM, latent_dim=LATENT_DIM,
    g_channels=G_CHANNELS, d_channels=D_CHANNELS, g_init_spatial=G_INIT_SPATIAL,
    g_params=int(generator.count_params()), d_params=int(discriminator.count_params()),
    g_lr=G_LR, d_lr=D_LR, beta1=ADAM_BETA1, beta2=ADAM_BETA2,
    global_batch_size=GLOBAL_BATCH_SIZE,
    epochs_completed=len(hist_df) if len(hist_df) else 0,
    target_epochs=TARGET_EPOCHS, best_epoch=best_epoch, best_g_loss=best_g_loss,
    final_g_loss=float(hist_df["g_loss"].iloc[-1]) if len(hist_df) else None,
    final_d_loss=float(hist_df["d_loss"].iloc[-1]) if len(hist_df) else None,
    zip_path=ZIP_PATH,
    zip_size_mb=round(os.path.getsize(ZIP_PATH)/1e6,1) if os.path.exists(ZIP_PATH) else None,
    tf_version=tf.__version__, num_gpus=len(gpus),
    generated_at=datetime.utcnow().isoformat()
)
with open(f"{LOG_DIR}/final_report.json","w") as f: json.dump(report,f,indent=2)

print("="*60)
print("DCGAN 280K — FINAL EXPERIMENT REPORT")
print("="*60)
for k,v in report.items(): print(f"  {k:<28}: {v}")
print("="*60)
print(f"Artifact ZIP: {ZIP_PATH}")
print("All done. Download the ZIP from the Kaggle Output tab.")